In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
%matplotlib inline
warnings.filterwarnings('ignore')

## 1. Cargar y Explorar los Datos

In [ ]:
# Crear datos sintéticos de pasajeros aéreos (similar al clásico AirPassengers dataset)
np.random.seed(42)

# Crear serie temporal de 144 meses (12 años)
meses = pd.date_range(start='1949-01-01', periods=144, freq='MS')

# Generar tendencia, estacionalidad y ruido
tendencia = np.linspace(100, 400, 144)
estacionalidad = 50 * np.sin(np.arange(144) * 2 * np.pi / 12)
ruido = np.random.normal(0, 20, 144)

# Combinar componentes
pasajeros = tendencia + estacionalidad + ruido
pasajeros = pasajeros.clip(min=50)  # Asegurar valores positivos

# Crear DataFrame
df = pd.DataFrame({
    'Fecha': meses,
    'Pasajeros': pasajeros
})
df.set_index('Fecha', inplace=True)

print("Dimensiones del dataset:", df.shape)
print("\nPrimeras filas:")
print(df.head(12))
print("\nÚltimas filas:")
print(df.tail(12))

In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas:")
print(df.describe())

print(f"\nPeriodo de datos: {df.index.min().strftime('%B %Y')} - {df.index.max().strftime('%B %Y')}")
print(f"Total de observaciones: {len(df)} meses")

In [ ]:
# Visualizar la serie temporal completa
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['Pasajeros'], linewidth=2, color='steelblue')
plt.xlabel('Año')
plt.ylabel('Número de Pasajeros (miles)')
plt.title('Serie Temporal de Pasajeros Aéreos Mensuales')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Análisis de Componentes de la Serie Temporal

In [ ]:
# Descomponer la serie en tendencia, estacionalidad y residuo
decomposition = seasonal_decompose(df['Pasajeros'], model='additive', period=12)

# Visualizar los componentes
fig, axes = plt.subplots(4, 1, figsize=(14, 10))

# Serie original
decomposition.observed.plot(ax=axes[0], color='steelblue')
axes[0].set_ylabel('Original')
axes[0].set_title('Descomposición de la Serie Temporal')

# Tendencia
decomposition.trend.plot(ax=axes[1], color='green')
axes[1].set_ylabel('Tendencia')

# Estacionalidad
decomposition.seasonal.plot(ax=axes[2], color='orange')
axes[2].set_ylabel('Estacionalidad')

# Residuo
decomposition.resid.plot(ax=axes[3], color='red')
axes[3].set_ylabel('Residuo')
axes[3].set_xlabel('Año')

plt.tight_layout()
plt.show()

## 3. Prueba de Estacionariedad (Test de Dickey-Fuller)

In [ ]:
# Realizar test de Dickey-Fuller Aumentado (ADF)
def test_estacionariedad(serie, nombre='Serie'):
    resultado = adfuller(serie, autolag='AIC')
    
    print(f'\nResultados del Test ADF para {nombre}:')
    print(f'Estadístico ADF: {resultado[0]:.6f}')
    print(f'p-valor: {resultado[1]:.6f}')
    print(f'Valores críticos:')
    for key, value in resultado[4].items():
        print(f'  {key}: {value:.3f}')
    
    if resultado[1] <= 0.05:
        print(f"\n✓ La serie ES ESTACIONARIA (p-valor ≤ 0.05)")
    else:
        print(f"\n✗ La serie NO ES ESTACIONARIA (p-valor > 0.05)")
    
    return resultado[1] <= 0.05

# Test en la serie original
es_estacionaria = test_estacionariedad(df['Pasajeros'], 'Serie Original')

In [ ]:
# Si no es estacionaria, aplicar diferenciación
if not es_estacionaria:
    # Primera diferencia
    df['Pasajeros_diff'] = df['Pasajeros'].diff()
    
    # Visualizar serie diferenciada
    plt.figure(figsize=(14, 5))
    plt.plot(df.index[1:], df['Pasajeros_diff'][1:], linewidth=2, color='coral')
    plt.xlabel('Año')
    plt.ylabel('Diferencia de Pasajeros')
    plt.title('Serie Temporal Diferenciada (d=1)')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Test en la serie diferenciada
    test_estacionariedad(df['Pasajeros_diff'].dropna(), 'Serie Diferenciada')

## 4. Determinar Parámetros p y q (ACF y PACF)

In [ ]:
# Gráficos ACF y PACF para determinar parámetros
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ACF - Autocorrelación (ayuda a determinar q)
plot_acf(df['Pasajeros'].dropna(), lags=40, ax=axes[0])
axes[0].set_title('Función de Autocorrelación (ACF)')
axes[0].set_xlabel('Lag')

# PACF - Autocorrelación Parcial (ayuda a determinar p)
plot_pacf(df['Pasajeros'].dropna(), lags=40, ax=axes[1])
axes[1].set_title('Función de Autocorrelación Parcial (PACF)')
axes[1].set_xlabel('Lag')

plt.tight_layout()
plt.show()

print("Interpretación:")
print("- ACF: Los lags significativos sugieren el orden q (MA)")
print("- PACF: Los lags significativos sugieren el orden p (AR)")

## 5. Preparar Datos de Entrenamiento y Prueba

In [ ]:
# Dividir en 80% entrenamiento, 20% prueba
train_size = int(len(df) * 0.8)
train_data = df['Pasajeros'][:train_size]
test_data = df['Pasajeros'][train_size:]

print(f"Datos de entrenamiento: {len(train_data)} observaciones")
print(f"  Periodo: {train_data.index.min().strftime('%B %Y')} - {train_data.index.max().strftime('%B %Y')}")
print(f"\nDatos de prueba: {len(test_data)} observaciones")
print(f"  Periodo: {test_data.index.min().strftime('%B %Y')} - {test_data.index.max().strftime('%B %Y')}")

# Visualizar la división
plt.figure(figsize=(14, 6))
plt.plot(train_data.index, train_data, label='Entrenamiento', linewidth=2, color='steelblue')
plt.plot(test_data.index, test_data, label='Prueba', linewidth=2, color='orange')
plt.xlabel('Año')
plt.ylabel('Número de Pasajeros (miles)')
plt.title('División de Datos: Entrenamiento vs Prueba')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Entrenar el Modelo ARIMA

In [ ]:
# Crear y entrenar el modelo ARIMA
# Parámetros (p, d, q):
#   p = orden autoregresivo (AR)
#   d = grado de diferenciación (I)
#   q = orden de media móvil (MA)

# Comenzamos con ARIMA(1, 1, 1) - un modelo común de inicio
p, d, q = 1, 1, 1

print(f"Entrenando modelo ARIMA({p}, {d}, {q})...")

# Crear y ajustar el modelo
arima_model = ARIMA(train_data, order=(p, d, q))
arima_fit = arima_model.fit()

print("\nModelo entrenado exitosamente")
print("\nResumen del modelo:")
print(arima_fit.summary())

## 7. Realizar Predicciones

In [ ]:
# Hacer predicciones en el conjunto de prueba
predicciones = arima_fit.forecast(steps=len(test_data))

# Crear DataFrame con resultados
resultados = pd.DataFrame({
    'Real': test_data.values,
    'Predicho': predicciones.values
}, index=test_data.index)

print("Ejemplos de predicciones:")
print(resultados.head(10))

## 8. Evaluar el Modelo

In [ ]:
# Calcular métricas de error
mse = mean_squared_error(test_data, predicciones)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_data, predicciones)
mape = np.mean(np.abs((test_data - predicciones) / test_data)) * 100

print("Métricas de Evaluación:")
print(f"MSE (Error Cuadrático Medio): {mse:.2f}")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse:.2f}")
print(f"MAE (Error Absoluto Medio): {mae:.2f}")
print(f"MAPE (Error Porcentual Absoluto Medio): {mape:.2f}%")

## 9. Visualizar Predicciones

In [ ]:
# Visualizar predicciones vs valores reales
plt.figure(figsize=(14, 6))

# Datos de entrenamiento
plt.plot(train_data.index, train_data, label='Datos de Entrenamiento', 
         linewidth=2, color='steelblue')

# Datos reales de prueba
plt.plot(test_data.index, test_data, label='Datos Reales (Prueba)', 
         linewidth=2, color='green')

# Predicciones
plt.plot(test_data.index, predicciones, label='Predicciones ARIMA', 
         linewidth=2, color='red', linestyle='--')

plt.xlabel('Año')
plt.ylabel('Número de Pasajeros (miles)')
plt.title(f'Predicciones del Modelo ARIMA({p},{d},{q})\nRMSE: {rmse:.2f}, MAPE: {mape:.2f}%')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom en el periodo de prueba
plt.figure(figsize=(14, 6))
plt.plot(test_data.index, test_data, label='Valores Reales', 
         linewidth=2, marker='o', color='green')
plt.plot(test_data.index, predicciones, label='Predicciones', 
         linewidth=2, marker='s', color='red', linestyle='--')
plt.xlabel('Fecha')
plt.ylabel('Número de Pasajeros (miles)')
plt.title('Comparación Detallada: Valores Reales vs Predicciones')
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 10. Análisis de Residuos

In [ ]:
# Analizar los residuos del modelo
residuos = arima_fit.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Serie de residuos
residuos.plot(ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Residuos del Modelo')
axes[0, 0].set_ylabel('Residuo')
axes[0, 0].axhline(y=0, color='red', linestyle='--')

# Histograma de residuos
residuos.hist(bins=30, ax=axes[0, 1], edgecolor='black')
axes[0, 1].set_title('Distribución de Residuos')
axes[0, 1].set_xlabel('Residuo')
axes[0, 1].set_ylabel('Frecuencia')

# Q-Q plot
from scipy import stats
stats.probplot(residuos, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot')

# ACF de residuos
plot_acf(residuos, lags=30, ax=axes[1, 1])
axes[1, 1].set_title('ACF de Residuos')

plt.tight_layout()
plt.show()

print("Estadísticas de los residuos:")
print(f"Media: {residuos.mean():.4f} (debe estar cerca de 0)")
print(f"Desviación estándar: {residuos.std():.4f}")

## 11. Predicción Futura

In [ ]:
# Reentrenar el modelo con todos los datos disponibles
modelo_final = ARIMA(df['Pasajeros'], order=(p, d, q))
modelo_final_fit = modelo_final.fit()

# Predecir los próximos 12 meses
n_periodos_futuros = 12
prediccion_futura = modelo_final_fit.forecast(steps=n_periodos_futuros)

# Crear índice de fechas futuras
ultima_fecha = df.index[-1]
fechas_futuras = pd.date_range(start=ultima_fecha + pd.DateOffset(months=1), 
                                periods=n_periodos_futuros, freq='MS')

# Visualizar
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['Pasajeros'], label='Datos Históricos', 
         linewidth=2, color='steelblue')
plt.plot(fechas_futuras, prediccion_futura, label='Predicción Futura', 
         linewidth=2, color='red', linestyle='--', marker='o')
plt.xlabel('Año')
plt.ylabel('Número de Pasajeros (miles)')
plt.title(f'Predicción de Pasajeros para los Próximos {n_periodos_futuros} Meses')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Mostrar predicciones futuras
print("Predicciones para los próximos meses:")
predicciones_df = pd.DataFrame({
    'Fecha': fechas_futuras,
    'Pasajeros Predichos': prediccion_futura.values
})
print(predicciones_df)

## Conclusiones

- ARIMA es un modelo potente para predecir series de tiempo con tendencia y estacionalidad
- Los parámetros (p, d, q) deben seleccionarse cuidadosamente usando ACF, PACF y pruebas de estacionariedad
- El modelo captura patrones temporales y permite hacer predicciones futuras
- Es importante validar que los residuos se comporten como ruido blanco (media 0, sin autocorrelación)
- Para series con estacionalidad fuerte, considerar SARIMA (ARIMA estacional)